# U-Net training template (Google Colab GPU)

This notebook is adapted for your CVAT-style dataset and is fully self-contained.
- Images are read from `dataset-preprocessed/...` in Google Drive
- Masks are built directly from `annotations.xml` inside this notebook (no `api.py` needed)

### What you need in Drive
- `dataset-preprocessed/` at the root of MyDrive
- `annotations.xml` at the root of MyDrive

In [2]:
# Colab deps (uncomment if needed)
# !pip -q install opencv-python-headless==4.10.0.84

import random
from pathlib import Path
import xml.etree.ElementTree as ET

import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

In [3]:
# Runtime setup: GPU + optional Google Drive mount
from google.colab import drive

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("GPU not found, using CPU")

MOUNT_DRIVE = True
if MOUNT_DRIVE:
    drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/a

MyDrive


In [ ]:
# ---- PATHS FOR YOUR DATA IN GOOGLE DRIVE ----
DRIVE_ROOT = Path('/content/drive/MyDrive')
DATASET_ROOT = DRIVE_ROOT / 'dataset-preprocessed'
ANNOTATIONS_XML = DRIVE_ROOT / 'annotations.xml'
IMAGES_ROOT = DRIVE_ROOT

# Optional label filter from CVAT XML (None = all labels)
LABEL_FILTER = None

OUTPUT_DIR = DRIVE_ROOT / 'unet_runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 0  # Colab + Google Drive is usually more stable with 0
EPOCHS = 20
LR = 3e-4
VAL_SPLIT = 0.2
CACHE_SAMPLES_IN_RAM = True

assert DATASET_ROOT.exists(), f'Missing dataset-preprocessed folder: {DATASET_ROOT}'
assert ANNOTATIONS_XML.exists(), f'Missing annotations.xml: {ANNOTATIONS_XML}'


def _parse_points(raw: str):
    points = []
    for pair in raw.split(';'):
        pair = pair.strip()
        if not pair:
            continue
        x, y = pair.split(',')
        points.append((float(x), float(y)))
    return points


def _polygons_to_mask(width: int, height: int, polygons: list[list[tuple[float, float]]]):
    mask = np.zeros((height, width), dtype=np.uint8)
    pts_all = []
    for poly in polygons:
        if len(poly) < 3:
            continue
        pts = np.round(np.array(poly, dtype=np.float32)).astype(np.int32)
        pts[:, 0] = np.clip(pts[:, 0], 0, width - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, height - 1)
        pts_all.append(pts)
    if pts_all:
        cv2.fillPoly(mask, pts_all, color=1)
    return mask


def _decode_cvat_rle(rle: str, width: int, height: int):
    vals = [int(v.strip()) for v in rle.split(',') if v.strip()]
    flat = np.zeros(width * height, dtype=np.uint8)
    idx, value = 0, 0
    for run in vals:
        if run <= 0:
            value = 1 - value
            continue
        end = min(idx + run, flat.size)
        if value == 1:
            flat[idx:end] = 1
        idx = end
        value = 1 - value
        if idx >= flat.size:
            break
    return flat.reshape((height, width))


def build_xml_index(xml_path: Path, label: str | None = None):
    index = {}
    for _event, image_elem in ET.iterparse(xml_path, events=('end',)):
        if image_elem.tag != 'image':
            continue

        image_name = image_elem.get('name', '')
        width = int(image_elem.get('width', '0'))
        height = int(image_elem.get('height', '0'))

        polygons = []
        for polygon in image_elem.findall('polygon'):
            poly_label = polygon.get('label', '')
            if label is not None and poly_label != label:
                continue
            polygons.append(_parse_points(polygon.get('points', '')))

        masks = []
        for mask in image_elem.findall('mask'):
            mask_label = mask.get('label', '')
            if label is not None and mask_label != label:
                continue
            masks.append({
                'rle': mask.get('rle', ''),
                'left': int(mask.get('left', '0')),
                'top': int(mask.get('top', '0')),
                'width': int(mask.get('width', '0')),
                'height': int(mask.get('height', '0')),
            })

        if polygons or masks:
            index[image_name] = {
                'width': width,
                'height': height,
                'polygons': polygons,
                'masks': masks,
            }

        image_elem.clear()

    return index


def get_image_and_mask_from_index(image_name: str, xml_index: dict, images_root: Path):
    info = xml_index[image_name]
    width, height = info['width'], info['height']

    image_path = images_root / image_name
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f'Failed to load image: {image_path}')

    full_mask = _polygons_to_mask(width, height, info['polygons'])

    for m in info['masks']:
        if m['width'] <= 0 or m['height'] <= 0:
            continue
        local_mask = _decode_cvat_rle(m['rle'], m['width'], m['height'])
        y1, x1 = m['top'], m['left']
        y2, x2 = y1 + m['height'], x1 + m['width']

        y1c, x1c = max(0, y1), max(0, x1)
        y2c, x2c = min(height, y2), min(width, x2)
        if y1c >= y2c or x1c >= x2c:
            continue

        ly1, lx1 = y1c - y1, x1c - x1
        ly2, lx2 = ly1 + (y2c - y1c), lx1 + (x2c - x1c)
        full_mask[y1c:y2c, x1c:x2c] = np.maximum(full_mask[y1c:y2c, x1c:x2c], local_mask[ly1:ly2, lx1:lx2])

    return image, full_mask


xml_index = build_xml_index(ANNOTATIONS_XML, label=LABEL_FILTER)
image_names = [name for name in sorted(xml_index.keys()) if (IMAGES_ROOT / name).exists()]

assert len(image_names) > 0, 'No valid annotated image paths found.'
print(f'Annotated + usable files: {len(image_names)}')

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(
        self,
        image_names: list[str],
        xml_index: dict,
        images_root: Path,
        img_size: int = 256,
        cache_in_ram: bool = True,
    ):
        self.image_names = image_names
        self.xml_index = xml_index
        self.images_root = images_root
        self.img_size = img_size
        self.cache_in_ram = cache_in_ram
        self._cache: dict[str, tuple[torch.Tensor, torch.Tensor]] = {}

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx: int):
        image_name = self.image_names[idx]

        if self.cache_in_ram and image_name in self._cache:
            return self._cache[image_name]

        image, mask = get_image_and_mask_from_index(image_name, self.xml_index, self.images_root)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask.astype(np.uint8), (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        image = image.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)

        image_t = torch.from_numpy(image).permute(2, 0, 1)  # [C,H,W]
        mask_t = torch.from_numpy(mask).unsqueeze(0)        # [1,H,W]

        if self.cache_in_ram:
            self._cache[image_name] = (image_t, mask_t)

        return image_t, mask_t

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=(64, 128, 256, 512)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f))
            ch = f

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        rev = features[::-1]
        up_in = features[-1] * 2
        for f in rev:
            self.ups.append(nn.ConvTranspose2d(up_in, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f * 2, f))
            up_in = f

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skips[idx // 2]

            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)

            x = torch.cat([skip, x], dim=1)
            x = self.ups[idx + 1](x)

        return self.final_conv(x)

In [ ]:
def dice_loss(logits, targets, smooth=1.0):
    probs = torch.sigmoid(logits)
    probs = probs.flatten(1)
    targets = targets.flatten(1)
    intersection = (probs * targets).sum(dim=1)
    denom = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + smooth) / (denom + smooth)
    return 1.0 - dice.mean()


def bce_dice_loss(logits, targets, bce_w=0.5, dice_w=0.5):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    d = dice_loss(logits, targets)
    return bce_w * bce + dice_w * d


@torch.no_grad()
def dice_score(logits, targets, thr=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs >= thr).float().flatten(1)
    targets = targets.flatten(1)
    tp = (preds * targets).sum(dim=1)
    fp = (preds * (1 - targets)).sum(dim=1)
    fn = ((1 - preds) * targets).sum(dim=1)
    return ((2 * tp + eps) / (2 * tp + fp + fn + eps)).mean().item()


def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)

    total_loss = 0.0
    total_dice = 0.0

    for images, masks in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = bce_dice_loss(logits, masks)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        total_dice += dice_score(logits.detach(), masks)

    n = max(1, len(loader))
    return total_loss / n, total_dice / n

In [ ]:
dataset = SegmentationDataset(
    image_names=image_names,
    xml_index=xml_index,
    images_root=IMAGES_ROOT,
    img_size=IMG_SIZE,
    cache_in_ram=CACHE_SAMPLES_IN_RAM,
)

val_size = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)
if NUM_WORKERS > 0:
    loader_kwargs['persistent_workers'] = True

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)

# quick loader sanity-check (helps catch stalls before training loop)
sample_images, sample_masks = next(iter(train_loader))
print('First batch:', tuple(sample_images.shape), tuple(sample_masks.shape))

model = UNet(in_channels=3, out_channels=1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_val_dice = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_dice = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_dice = run_epoch(model, val_loader, optimizer=None)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_dice': train_dice,
        'val_loss': val_loss,
        'val_dice': val_dice,
    })

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_dice={train_dice:.4f} | "
        f"val_loss={val_loss:.4f} val_dice={val_dice:.4f}"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_unet.pt')

print(f'Finished. Best val dice: {best_val_dice:.4f}')
print(f'Model saved to: {OUTPUT_DIR / "best_unet.pt"}')

In [ ]:
# Optional: quick prediction preview on one validation sample
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    sample_img, sample_mask = val_ds[0]
    logits = model(sample_img.unsqueeze(0).to(device))
    pred = (torch.sigmoid(logits).squeeze().cpu().numpy() > 0.5).astype(np.uint8)

img_np = sample_img.permute(1, 2, 0).numpy()
mask_np = sample_mask.squeeze(0).numpy()

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title('Image')
plt.imshow(img_np)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title('GT Mask')
plt.imshow(mask_np, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('Pred Mask')
plt.imshow(pred, cmap='gray')
plt.axis('off')
plt.show()